In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [2]:
df=pd.read_csv('../data/healthcare_cleaned.csv')

In [6]:
drop_cols=['patient_id','treatment_cost','mortality_risk','diagnosis_category','length_of_stay','readmission_30day']

In [7]:
X=df.drop(drop_cols,axis=1)
y=df['mortality_risk']

In [10]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)


In [20]:
from sklearn.metrics import classification_report,confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
cat_cols=[col for col in X.columns if X[col].dtypes=='str']
num_cols=[col for col in X.columns if col not in cat_cols]
Preprocessor=ColumnTransformer(transformers=[
    ('num',StandardScaler(),num_cols),
    ('cat',OneHotEncoder(handle_unknown='ignore'),cat_cols)
])
smote_pipeline=ImbPipeline(steps=[
    ('Preprocessor',Preprocessor),
    ('smote',SMOTE(random_state=42)),
    ('model',DecisionTreeClassifier(random_state=42))
])
smote_pipeline.fit(X_train,y_train)
y_pred=smote_pipeline.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.94      0.92      0.93      9430
           1       0.07      0.09      0.07       570

    accuracy                           0.88     10000
   macro avg       0.50      0.51      0.50     10000
weighted avg       0.89      0.88      0.88     10000



In [28]:
from sklearn.model_selection import GridSearchCV
params={
    'model__max_depth':[3,5,7],
    'model__min_samples_split':[5,10,15]
}
grid_model_df=GridSearchCV(n_jobs=-1,estimator=smote_pipeline,param_grid=params,cv=5,scoring='f1')
grid_model_df.fit(X_train,y_train)
y_pred=grid_model_df.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.95      0.65      0.77      9430
           1       0.07      0.42      0.12       570

    accuracy                           0.64     10000
   macro avg       0.51      0.54      0.45     10000
weighted avg       0.90      0.64      0.74     10000



In [29]:
print(grid_model_df.best_params_)
print(grid_model_df.best_score_)

{'model__max_depth': 3, 'model__min_samples_split': 5}
0.12242100058293855


In [30]:
smote_pipeline_rf=ImbPipeline(steps=[
    ('preprocessor',Preprocessor),
    ('smote',SMOTE(random_state=42)),
    ('model',RandomForestClassifier(random_state=42))
])
smote_pipeline_rf.fit(X_train,y_train)
y_pred=smote_pipeline_rf.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.94      1.00      0.97      9430
           1       0.00      0.00      0.00       570

    accuracy                           0.94     10000
   macro avg       0.47      0.50      0.49     10000
weighted avg       0.89      0.94      0.92     10000



c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [32]:
params_rf={
    'model__n_estimators':[10,50,100,150],
    'model__max_depth':[3,5,7,9]
}
grid_model_rf=GridSearchCV(estimator=smote_pipeline_rf,scoring='f1',n_jobs=-1,param_grid=params_rf,cv=5)
grid_model_rf.fit(X_train,y_train)
y_pred=grid_model_rf.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.95      0.77      0.85      9430
           1       0.08      0.34      0.13       570

    accuracy                           0.74     10000
   macro avg       0.52      0.55      0.49     10000
weighted avg       0.90      0.74      0.81     10000



In [33]:
print(grid_model_rf.best_params_)
print(grid_model_rf.best_score_)

{'model__max_depth': 3, 'model__n_estimators': 10}
0.13560817883977555


In [34]:
scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
print(scale_pos_weight)

16.54385964912281


In [36]:
smote_pipeline_xgb=ImbPipeline(steps=[
    ('preprocessor',Preprocessor),
    ('smote',SMOTE(random_state=42)),
    ('model',XGBClassifier(random_state=42,eval_metric='logloss'))
])
smote_pipeline_xgb.fit(X_train,y_train)
y_pred=smote_pipeline_xgb.predict(X_test)
print(classification_report(y_test,y_pred))
params_xgb={
    'model__n_estimators':[10,50,100,150],
    'model__max_depth':[3,5,7,9],
    'model__learning_rate':[0.01,0.05,0.1,0.3]
}
grid_model_xgb=GridSearchCV(estimator=smote_pipeline_xgb,scoring='f1',n_jobs=-1,param_grid=params_xgb,cv=5)
grid_model_xgb.fit(X_train,y_train)
y_pred=grid_model_xgb.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.94      1.00      0.97      9430
           1       0.00      0.00      0.00       570

    accuracy                           0.94     10000
   macro avg       0.47      0.50      0.49     10000
weighted avg       0.89      0.94      0.91     10000

              precision    recall  f1-score   support

           0       0.95      0.76      0.84      9430
           1       0.06      0.27      0.10       570

    accuracy                           0.74     10000
   macro avg       0.51      0.52      0.47     10000
weighted avg       0.90      0.74      0.80     10000



In [38]:
smote_pipeline_xgb=ImbPipeline(steps=[
    ('preprocessor',Preprocessor),
    ('model',XGBClassifier(random_state=42,eval_metric='logloss',scale_pos_weight=scale_pos_weight))
])
smote_pipeline_xgb.fit(X_train,y_train)
y_pred=smote_pipeline_xgb.predict(X_test)
print(classification_report(y_test,y_pred))
params_xgb={
    'model__n_estimators':[10,50,100,150],
    'model__max_depth':[3,5,7,9],
    'model__learning_rate':[0.01,0.05,0.1,0.3]
}
grid_model_xgb=GridSearchCV(estimator=smote_pipeline_xgb,scoring='f1',n_jobs=-1,param_grid=params_xgb,cv=5)
grid_model_xgb.fit(X_train,y_train)
y_pred=grid_model_xgb.predict(X_test)
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.95      0.89      0.92      9430
           1       0.08      0.15      0.10       570

    accuracy                           0.85     10000
   macro avg       0.51      0.52      0.51     10000
weighted avg       0.90      0.85      0.87     10000

              precision    recall  f1-score   support

           0       0.96      0.66      0.78      9430
           1       0.09      0.52      0.15       570

    accuracy                           0.66     10000
   macro avg       0.52      0.59      0.47     10000
weighted avg       0.91      0.66      0.75     10000



In [39]:
print(grid_model_xgb.best_params_)
print(grid_model_xgb.best_score_)

{'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 10}
0.1527474612127782


In [47]:
# threshold tuning
y_pred_proba=grid_model_xgb.predict_proba(X_test)[:,1]
y_pred_threshold=(y_pred_proba>0.3).astype(int)
print(classification_report(y_test,y_pred_threshold))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00      9430
           1       0.06      1.00      0.11       570

    accuracy                           0.06     10000
   macro avg       0.03      0.50      0.05     10000
weighted avg       0.00      0.06      0.01     10000



c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\admin\Health_Care\venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [48]:

print(pd.Series(y_pred_proba).describe())

count    10000.000000
mean         0.496970
std          0.012140
min          0.478082
25%          0.485776
50%          0.499729
75%          0.507683
max          0.518762
dtype: float64


Observations:
1) Best model: XGBoost + scale_pos_weight, F1=0.15, recall=0.52
2) All predicted probabilities clustered between 0.47-0.52 — 
   model has no confidence in any prediction,estinally guessing every prediction
3) Threshold tuning ineffective — as there is no probability spread
4) Root cause: features in this dataset carry insufficient signal 
   for mortality prediction , even smote re sample failed to improve

In [49]:
import joblib
joblib.dump(grid_model_xgb,'../models/mortality_model.pkl')

['../models/mortality_model.pkl']